# Class-specific SHAP importance for selected vegetation classes

This notebook complements the global impurity-based (MDI) variable ranking of the random forest vegetation classifier with **class-specific** importance from SHAP values. It is intended for examining class pairs that the classifier frequently confuses: if two confused classes rely on the same top predictors, the available predictors do not separate them well.

**What it does**

1. Reads the labelled segments (training and validation plots) from the project geopackage.
2. For each run `i` (0 … `N_RUNS`−1), draws a stratified random training/validation split (70% of each class for training, seed `i`) and trains an unweighted random forest (`random_state = i`). This is the same split and seed scheme used in the repeated random-split notebook, so run `i` here reproduces run `i` there.
3. Computes SHAP values (`shap.TreeExplainer`) for the validation segments of the selected classes and, for each class, the mean absolute SHAP value of every predictor **for that class's output, over that class's own validation segments**.
4. Averages the per-class rankings across runs and records how often each predictor is among a class's top five, as a measure of ranking stability.

Results are saved after every run, so an interrupted run can be resumed by re-running the notebook.

**Requirements:** Python 3.10+, `geopandas` (with `pyogrio` or `fiona`), `pandas`, `numpy`, `scikit-learn`, `shap`.

**Data:** the field-plot and imagery-derived data are not distributed with this repository. Set `GEOPACKAGE` below to point to your own copy. The layer whose name contains `MergedSegments` must contain a `Class` column, a `training_type` column (`training` / `truth` for labelled segments) and the predictor columns listed in `FEATURES`.

**Interpretation notes:** SHAP, like MDI, divides credit among correlated predictors (for example the mean and standard deviation of the same layer, or neighbouring principal components), so closely related predictors should be read together. Rankings for classes with few validation segments are less stable; the top-five frequency columns show how consistent each ranking is across runs.

In [ ]:
import os
import json
import time
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import shap
from sklearn.ensemble import RandomForestClassifier

warnings.simplefilter(action='ignore', category=FutureWarning)
print('shap', shap.__version__)

## Settings

In [ ]:
# Paths are relative to the folder containing this notebook.
NOTEBOOK_DIR = os.path.abspath('')
GEOPACKAGE = os.path.join(NOTEBOOK_DIR, 'pilot_project_RFv12.1.gpkg')        # input data (not distributed)
OUTPUT_DIR = os.path.join(os.path.dirname(NOTEBOOK_DIR), 'rf_outputs', 'shap_class_pairs')

N_RUNS = 100            # number of repeated random splits; run i uses seed i for the split and the forest
TRAIN_FRACTION = 0.7    # fraction of each class used for training
CLASS_WEIGHT = None     # None (unweighted), "balanced" or "balanced_subsample"
TOP_N = 5               # number of top predictors reported per class

# Classes to analyse, grouped as (label, [class codes]); edit as needed.
CLASS_GROUPS = [
    ('Mixed-conifer', [1310, 1320]),
    ('Pinyon-juniper woodland', [2111, 2212]),
    ('Semi-desert grassland', [4411, 4431]),
    ('Montane shrubland', [3112, 3111]),
    ('Reference (well separated)', [1111]),
]
CLASSES_OF_INTEREST = [c for _, codes in CLASS_GROUPS for c in codes]

# Predictor variables (65), identical to the main random forest model.
FEATURES = ['GLCM_Homog', 'GLCM_Mean_', 'Mean_Aspec', 'Mean_Blue', 'Mean_DEM', 'Mean_gedi_', 'Mean_Green',
            'Mean_NDMI', 'Mean_NIR', 'Mean_Red', 'Mean_SolRa', 'Mean_TWI', 'Median_DHM',
            'Mn_smap_10', 'Mn_smap_c1', 'Mn_smap_c2', 'Mn_smap_c3', 'Mn_smap_c4', 'Mn_smap_c5', 'Mn_smap_c6',
            'Mn_smap_c7', 'Mn_smap_c8', 'Mn_smap_c9', 'R_Grass', 'R_high_tree', 'R_low_shrub', 'R_low_tree',
            'R_med_shrub', 'Rel_Bare', 'St_smap_10', 'Std_gedi', 'Std_NDMI', 'St_smap_c9', 'St_smap_c8',
            'St_smap_c7', 'St_smap_c6', 'St_smap_c2', 'St_smap_c5', 'St_smap_c4', 'St_smap_c3', 'St_smap_c1',
            'Std_Blue', 'Std_DEM', 'Std_DHM', 'Std_Green', 'Std_NIR', 'Std_Red', 'Std_SolRad', 'Std_TWI',
            'Mean_PCA1', 'Mean_PCA2', 'Mean_PCA3', 'Mean_PCA4', 'Mean_PCA5', 'Mean_PCA6', 'Mean_PCA7',
            'Mean_PCA8', 'Std_PCA1', 'Std_PCA2', 'Std_PCA3', 'Std_PCA4', 'Std_PCA5', 'Std_PCA6', 'Std_PCA7',
            'Std_PCA8']

os.makedirs(OUTPUT_DIR, exist_ok=True)
RUNS_FILE = os.path.join(OUTPUT_DIR, 'shap_by_run.csv')      # one row per run x class x predictor (checkpoint)
print('Output folder:', OUTPUT_DIR)

## Load the labelled segments

In [ ]:
if not os.path.exists(GEOPACKAGE):
    raise FileNotFoundError(f'Geopackage not found: {GEOPACKAGE}')

try:
    layer_names = [name for name, _ in gpd.list_layers(GEOPACKAGE).values]
except AttributeError:  # geopandas < 1.0
    import fiona
    layer_names = fiona.listlayers(GEOPACKAGE)
segment_layer = next((n for n in layer_names if 'mergedsegments' in n.lower()), None)
if segment_layer is None:
    raise ValueError('No layer containing "MergedSegments" found in the geopackage')

# Attributes only; reading a large geopackage can take a few minutes.
segment_df = gpd.read_file(GEOPACKAGE, layer=segment_layer, ignore_geometry=True)
labelled_df = segment_df.loc[segment_df['training_type'].isin(['training', 'truth']), FEATURES + ['Class']].copy()
labelled_df['Class'] = labelled_df['Class'].astype(int)

missing = [c for c in CLASSES_OF_INTEREST if c not in set(labelled_df['Class'])]
if missing:
    raise ValueError(f'Classes not found in the labelled data: {missing}')
print(f'{len(labelled_df)} labelled segments, {labelled_df.Class.nunique()} classes')

## Split, train and compute SHAP for each run

In [ ]:
def random_split(labelled_df, train_fraction, seed):
    """Stratified random split: for each class, sample train_fraction of the segments for training.
    Returns a Series of 'training' / 'truth' aligned to labelled_df."""
    split = pd.Series('truth', index=labelled_df.index)
    for class_id in sorted(labelled_df['Class'].unique()):
        this_class_rows = labelled_df[labelled_df['Class'] == class_id]
        training_rows = this_class_rows.sample(frac=train_fraction, random_state=seed)
        split.loc[training_rows.index] = 'training'
    return split


def class_shap_importance(model, X_truth, y_truth, classes):
    """Mean absolute SHAP value of each predictor for each class's own output,
    averaged over that class's validation segments. Returns a long DataFrame."""
    mask = np.isin(y_truth, classes)
    X = X_truth[mask]
    y = y_truth[mask]
    values = shap.TreeExplainer(model).shap_values(X, check_additivity=False)
    # shap >= 0.45 returns (samples, features, classes); older versions return a list per class
    if isinstance(values, list):
        values = np.stack(values, axis=-1)
    model_classes = list(model.classes_.astype(int))
    rows = []
    for c in classes:
        sel = (y == c)
        if not sel.any():
            continue
        mean_abs = np.abs(values[sel, :, model_classes.index(c)]).mean(axis=0)
        order = np.argsort(-mean_abs)
        ranks = np.empty(len(order), dtype=int)
        ranks[order] = np.arange(1, len(order) + 1)
        for f, v, rk in zip(X.columns, mean_abs, ranks):
            rows.append({'class': c, 'n_truth': int(sel.sum()), 'feature': f, 'mean_abs_shap': float(v), 'rank': int(rk)})
    return pd.DataFrame(rows)

In [ ]:
done_runs = set(pd.read_csv(RUNS_FILE)['run']) if os.path.exists(RUNS_FILE) else set()
if done_runs:
    print(f'Resuming: {len(done_runs)} runs already in {RUNS_FILE}')

t0 = time.time()
for run_i in range(N_RUNS):
    if run_i in done_runs:
        continue
    split = random_split(labelled_df, TRAIN_FRACTION, seed=run_i)
    train = labelled_df[split == 'training']
    truth = labelled_df[split == 'truth']
    model = RandomForestClassifier(class_weight=CLASS_WEIGHT, random_state=run_i, n_jobs=-1)
    model.fit(train[FEATURES], train['Class'])

    result = class_shap_importance(model, truth[FEATURES], truth['Class'].to_numpy(), CLASSES_OF_INTEREST)
    result.insert(0, 'run', run_i)
    result.to_csv(RUNS_FILE, mode='a', header=not os.path.exists(RUNS_FILE), index=False)
    print(f'run {run_i} done ({time.time() - t0:.0f} s)')

shap_runs = pd.read_csv(RUNS_FILE)
shap_runs = shap_runs[shap_runs['run'] < N_RUNS]
print(f"{shap_runs['run'].nunique()} runs available")

## Summarise across runs

In [ ]:
n_runs_done = shap_runs['run'].nunique()
summary = (shap_runs
           .assign(in_top_n=lambda d: d['rank'] <= TOP_N)
           .groupby(['class', 'feature'])
           .agg(mean_abs_shap=('mean_abs_shap', 'mean'),
                sd_abs_shap=('mean_abs_shap', 'std'),
                median_rank=('rank', 'median'),
                top_n_frequency=('in_top_n', 'mean'))
           .reset_index())
summary['rank_of_mean'] = summary.groupby('class')['mean_abs_shap'].rank(ascending=False, method='first').astype(int)
summary = summary.sort_values(['class', 'rank_of_mean'])
summary.to_csv(os.path.join(OUTPUT_DIR, 'shap_class_summary.csv'), index=False)

# Top-N table: predictors ranked by mean |SHAP| across runs; % = share of runs in which the predictor was in the class's top N
group_of = {c: g for g, codes in CLASS_GROUPS for c in codes}
n_truth = shap_runs.groupby('class')['n_truth'].first()
top_rows = []
for c in CLASSES_OF_INTEREST:
    s = summary[summary['class'] == c].head(TOP_N)
    row = {'Group': group_of[c], 'Class': c, 'Validation segments per run': int(n_truth.get(c, 0))}
    for k, (_, r) in enumerate(s.iterrows(), start=1):
        row[f'Rank {k}'] = f"{r['feature']} ({r['top_n_frequency'] * 100:.0f}%)"
    top_rows.append(row)
top_table = pd.DataFrame(top_rows)
top_table.to_csv(os.path.join(OUTPUT_DIR, f'shap_top{TOP_N}_by_class.csv'), index=False)

with open(os.path.join(OUTPUT_DIR, 'shap_settings.json'), 'w') as f:
    json.dump({'n_runs': int(n_runs_done), 'train_fraction': TRAIN_FRACTION, 'class_weight': CLASS_WEIGHT,
               'top_n': TOP_N, 'class_groups': CLASS_GROUPS, 'n_features': len(FEATURES),
               'shap_version': shap.__version__}, f, indent=2)
print(f'Summary over {n_runs_done} runs saved to {OUTPUT_DIR}')
top_table

## Shared top predictors within each group

For each group, the predictors that appear in the top-N list of every class in the group. Shared top predictors indicate overlapping predictor signatures that can contribute to confusion between the classes.

In [ ]:
shared_rows = []
for g, codes in CLASS_GROUPS:
    if len(codes) < 2:
        continue
    tops = [set(summary[summary['class'] == c].head(TOP_N)['feature']) for c in codes]
    shared = set.intersection(*tops)
    shared_rows.append({'Group': g, 'Classes': ' / '.join(map(str, codes)),
                        f'Shared top-{TOP_N} predictors': ', '.join(sorted(shared)) or '(none)'})
shared_table = pd.DataFrame(shared_rows)
shared_table.to_csv(os.path.join(OUTPUT_DIR, 'shap_shared_top_predictors.csv'), index=False)
shared_table